# 第四章：现代深度学习

在前三章中，我们从感知机出发，逐步认识了 CNN（图像）和 RNN/LSTM（序列）。但 RNN 存在一个根本性的瓶颈：必须顺序处理序列，且长距离信息容易在传递中衰减。

本章从 **RNN 的局限**出发，介绍解决这一问题的关键突破——**注意力机制（Attention）**和 **Transformer 架构**。Transformer 已成为当今几乎所有前沿模型（语言、图像、语音、多模态）的核心构件。在此基础上，本章依次介绍：

- **预训练模型与迁移学习**（BERT 和 HuggingFace 生态）
- **生成模型概览**（VAE、GAN、Diffusion Models 的原理对比）
- **大语言模型简介**（GPT vs BERT、Scaling Law、提示工程、RLHF）

通过本章，读者将建立对现代深度学习全貌的系统认识，并通过 IMDb 情感分类实战，直观感受 Transformer 相比 LSTM 的提升。

## 目录

| 章节 | 内容 |
|------|------|
| **4.1 注意力机制** | RNN 瓶颈、Q/K/V 直觉、Scaled Dot-Product Attention |
| **4.2 Transformer** | Multi-Head Attention、位置编码、Encoder 结构、情感分类实战 |
| **4.3 预训练与迁移学习** | BERT 架构、HuggingFace、与 LSTM 结果对比 |
| **4.4 生成模型概览** | VAE、GAN、Diffusion Models 对比 |
| **4.5 大语言模型简介** | GPT vs BERT、Scaling Law、提示工程、RLHF |

In [1]:
import math
import re
import pickle
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

matplotlib.rcParams['font.family'] = ['PingFang SC', 'Heiti TC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备：{DEVICE}')

使用设备：cpu


## 4.1 注意力机制（Attention）

### RNN 的信息瓶颈

Seq2Seq 模型（如神经机器翻译）由两部分组成：**Encoder** 读入源语言句子，将其压缩成一个固定长度的向量（即 LSTM 的最终隐状态）；**Decoder** 再从这个向量出发，逐词生成目标语言。

这个固定向量就是瓶颈所在——就像让人把一本书读完后，只能用**一句话**总结，再凭这一句话翻译全书。句子越长，信息丢失越严重。实验也证实：句子超过 30 个词后，Seq2Seq 的翻译质量断崖式下跌。

---

### Attention 的核心思想

Attention（注意力机制）的解法非常直觉：**解码每个词时，不依赖单一的压缩向量，而是动态地回头查看源句子的所有位置，根据相关性加权求和。**

- 翻译"猫"时，模型把更多注意力集中在源句中的"cat"位置
- 翻译"跑"时，注意力转向"ran"
- 每个解码步骤都能"回头看"源句，信息不再因压缩而丢失

这一思想由 Bahdanau 等人于 2015 年提出，是 NLP 领域的重要里程碑。

---

### Q / K / V 的直觉

**场景**：把英文句子 "I love machine learning" 翻译成中文"我爱机器学习"。

> **注意**：这里和 3.6 节的普通 Seq2Seq 不同。普通 Seq2Seq 的 Encoder 只把最后一步的隐状态（$c$）传给 Decoder，这就是信息瓶颈。加了 Attention 之后，Encoder **把每一步的隐状态全部传给 Decoder**，Decoder 在生成每个词时动态决定"重点看哪个位置"，不再受限于单个 $c$。
>
> ```
> 普通 Seq2Seq：  h₁  h₂  h₃  h₄ ──→ c ──→ Decoder    （只传最后一个）
> 加了 Attention：h₁  h₂  h₃  h₄ ──→ Decoder           （全部传递，动态查询）
> ```

现在 Encoder 已经把 4 个英文词各自编码成隐状态向量 $[h_1, h_2, h_3, h_4]$，Decoder 正在生成第二个中文词。它需要回头查询这 4 个隐状态，判断哪个词和"爱"最相关——这就是 Attention 要做的事。

Attention 把这个"查询信息"的过程拆成三个角色：

| 角色 | 含义 | 在这个翻译场景中 | 类比 |
|------|------|----------------|------|
| **Query（Q）** | 我现在在找什么？ | Decoder 当前步的隐状态（正在生成"爱"） | 搜索引擎的查询词 |
| **Key（K）** | 每个位置能提供什么？ | Encoder 4 个时间步的隐状态（I / love / machine / learning） | 文档的关键词索引 |
| **Value（V）** | 每个位置的实际内容 | 同上（同一批隐状态，但经过不同变换） | 文档的正文内容 |

**Q、K、V 从哪里来？** 三者都是对输入向量做**可学习的线性变换**得到的：

$$Q = h_{\text{decoder}} \cdot W^Q, \quad K = H_{\text{encoder}} \cdot W^K, \quad V = H_{\text{encoder}} \cdot W^V$$

$W^Q, W^K, W^V$ 是三组独立的权重矩阵，训练中模型自己学习如何把隐状态转换成好的查询向量、键向量和值向量。Q、K、V 的维度通常相同，记为 $d_k$。

**计算过程（三步）：**

```
场景：Decoder 正在生成"爱"，Query q 来自当前 Decoder 隐状态（已经过 W^Q 变换）

              英文源句:   I      love   machine  learning
              Key:       k₁      k₂       k₃       k₄

第 1 步：算相似度    q·k₁ᵀ  q·k₂ᵀ  q·k₃ᵀ  q·k₄ᵀ
（点积）              [0.3,    2.1,    0.5,    0.2]   ← 点积越大 = 越相关

第 2 步：Softmax     [0.06,   0.75,   0.11,   0.08]  ← 归一化为概率（注意力权重）
                              ↑
                  对"love"的注意力最高（0.75），符合直觉

第 3 步：加权求和   0.06·v₁ + 0.75·v₂ + 0.11·v₃ + 0.08·v₄
                   ≈ 以"love"位置的 Value 为主，混入少量其他位置信息
                   → 这个向量就是 Decoder 在生成"爱"时所用的"上下文信息"
```

Key 和 Value 来自同一组 Encoder 隐状态，但经过不同变换矩阵 $W^K$ 和 $W^V$：
- **Key** 负责"被检索"：决定这个位置与 Query 的相关性有多高
- **Value** 负责"被读取"：一旦被选中，实际输出的内容

---

### Scaled Dot-Product Attention

把上面三步写成矩阵形式（对所有 Query 位置一次性并行计算）：

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

- $QK^\top$：所有 Query 与所有 Key 的点积，形状 $(\text{seq}_q \times \text{seq}_k)$，对应第 1 步
- $\text{softmax}(\cdot)$：对每行归一化，得到注意力权重矩阵，对应第 2 步
- $(\cdot) V$：用权重对 Value 加权求和，对应第 3 步，输出形状 $(\text{seq}_q \times d_k)$

**为什么叫"Scaled Dot-Product"？** 这个名字描述的是机制本身——名字拆开来看：
- **Dot-Product**：Q 和 K 的相似度用**点积**来算（$QK^\top$）
- **Scaled**：点积之后除以 $\sqrt{d_k}$ 做了**缩放**

这是为了和更早的**加性注意力（Additive Attention，即 Bahdanau Attention，2015 年）**区分。加性注意力不用点积，而是把 Q 和 K 拼起来送进一个小型前馈网络来算相似度：

$$\text{score}(q, k) = v^\top \tanh(W_q q + W_k k)$$

所以"Scaled Dot-Product"的意思是：我用的是**加了缩放的点积**来算相似度，不是加性网络。

**为什么要除以 $\sqrt{d_k}$（缩放）？**

当 $d_k$ 较大时，$QK^\top$ 的点积量级随维度增大——想象两个 128 维随机向量，其点积的方差约为 128。量级过大会导致 softmax 的输入进入饱和区（某一项远大于其他项），输出接近 one-hot，梯度趋近于零，训练停滞。除以 $\sqrt{d_k}$ 将方差归一化回 1，保持 softmax 输出的「温度」适中。

**和 Multi-Head Attention 的关系**：Scaled Dot-Product Attention 是零件，Multi-Head Attention 是用这个零件组装出来的模块——它并行跑 $h$ 组独立的 Scaled Dot-Product Attention，每组学到不同维度的语义关系，最后把结果拼接起来。详见 4.2 节。

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# ── Scaled Dot-Product Attention 实现 ─────────────────────────────────────────
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (..., seq_q, d_k)
    K: (..., seq_k, d_k)
    V: (..., seq_k, d_v)
    返回：输出张量 (..., seq_q, d_v)，注意力权重 (..., seq_q, seq_k)
    """
    d_k = Q.size(-1)
    # 点积后除以 sqrt(d_k) 防止 softmax 饱和
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (..., seq_q, seq_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weights = F.softmax(scores, dim=-1)                              # (..., seq_q, seq_k)
    output  = torch.matmul(weights, V)                               # (..., seq_q, d_v)
    return output, weights


# ── 可视化 1：翻译对齐的注意力热力图 ──────────────────────────────────────────
src_tokens = ['I', 'love', 'machine', 'learning']
tgt_tokens = ['我', '爱', '机器', '学习']

# 手工设计一个对角线主导的对齐矩阵（体现词对词的对应关系）
align = np.array([
    [0.85, 0.08, 0.04, 0.03],   # 我  ← I
    [0.06, 0.82, 0.07, 0.05],   # 爱  ← love
    [0.04, 0.05, 0.86, 0.05],   # 机器 ← machine
    [0.03, 0.04, 0.06, 0.87],   # 学习 ← learning
])

# ── 可视化 2：Self-Attention 权重（随机向量） ─────────────────────────────────
seq_len, d_k = 6, 16
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)
_, self_attn = scaled_dot_product_attention(Q, K, V)
self_attn_np = self_attn.squeeze(0).detach().numpy()

# ── 并排绘图 ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('注意力权重可视化', fontsize=13, fontweight='bold')

# 图1：翻译对齐
ax = axes[0]
im = ax.imshow(align, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(src_tokens))); ax.set_xticklabels(src_tokens, fontsize=12)
ax.set_yticks(range(len(tgt_tokens))); ax.set_yticklabels(tgt_tokens, fontsize=12)
ax.set_xlabel('源语言（Source）', fontsize=11)
ax.set_ylabel('目标语言（Target）', fontsize=11)
ax.set_title('翻译对齐注意力\n（对角线主导 = 词对词对应）', fontsize=11)
for i in range(len(tgt_tokens)):
    for j in range(len(src_tokens)):
        ax.text(j, i, f'{align[i, j]:.2f}', ha='center', va='center',
                fontsize=10, color='black' if align[i, j] < 0.6 else 'white')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# 图2：Self-Attention
ax2 = axes[1]
im2 = ax2.imshow(self_attn_np, cmap='Purples', vmin=0, vmax=self_attn_np.max())
ax2.set_xticks(range(seq_len)); ax2.set_xticklabels([f'pos{i}' for i in range(seq_len)], fontsize=9)
ax2.set_yticks(range(seq_len)); ax2.set_yticklabels([f'pos{i}' for i in range(seq_len)], fontsize=9)
ax2.set_xlabel('Key 位置', fontsize=11)
ax2.set_ylabel('Query 位置', fontsize=11)
ax2.set_title(f'Self-Attention 权重矩阵\n（seq_len={seq_len}, d_k={d_k}，随机初始化）', fontsize=11)
for i in range(seq_len):
    for j in range(seq_len):
        ax2.text(j, i, f'{self_attn_np[i, j]:.2f}', ha='center', va='center',
                 fontsize=8, color='black' if self_attn_np[i, j] < 0.12 else 'white')
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print('Scaled Dot-Product Attention 公式：')
print('  Attention(Q, K, V) = softmax( Q·Kᵀ / √d_k ) · V')
print()
print(f'缩放因子 √d_k 的作用（以 d_k={d_k} 为例）：')
print(f'  未缩放时，点积的方差约为 d_k={d_k}（向量维度越大量级越大）')
print(f'  除以 √{d_k} = {math.sqrt(d_k):.2f} 后，方差归一化回 ~1')
print('  保持 softmax 输出温度适中，避免梯度消失')

## 4.2 Transformer

### 为什么抛弃 RNN？

RNN 有一个根本的并行化障碍：第 $t$ 步的计算依赖第 $t-1$ 步的隐状态，序列只能**顺序处理**。对于一句 512 词的句子，必须串行执行 512 步 LSTM 操作，无论硬件多强都没法加速。

Transformer（Vaswani 等，2017 年《Attention Is All You Need》）完全抛弃了递归结构，**用 Attention 一次性计算序列中所有位置对之间的关系**，计算完全可以并行。这使得在同等时间内，Transformer 可以处理数量级更多的数据，从而催生了大规模预训练的时代。

---

### Transformer 完整架构图

原始 Transformer（用于机器翻译）由 Encoder 和 Decoder 两部分组成：

```
        ENCODER                              DECODER
─────────────────────────          ──────────────────────────────
输入：源语言                         输入：已生成的目标语言词

Input Embedding                    Output Embedding
Positional Encoding                Positional Encoding
         │                                  │
┌────────▼────────┐  ┐             ┌────────▼──────────────┐  ┐
│ Self-Attention  │  │             │ Masked Self-Attention  │  │
│ + 残差 + LNorm  │  │             │ + 残差 + LNorm         │  │
├─────────────────┤  │× N          ├───────────────────────┤  │× N
│ Feed-Forward    │  │             │ Cross-Attention        │  │
│ + 残差 + LNorm  │  │             │ （Q←Decoder,K/V←Enc）  │  │
└────────┬────────┘  ┘             ├───────────────────────┤  │
         │                         │ Feed-Forward           │  │
         │                         │ + 残差 + LNorm         │  │
         └──────────────────────→  └──────────┬─────────────┘  ┘
              K / V 传给 Decoder               │
                                    Linear + Softmax
                                         │
                                      输出词
```

Decoder 比 Encoder 多两处设计：
- **Masked Self-Attention**：生成第 $t$ 个词时，只能看到前 $t-1$ 个已生成的词，不能"偷看"未来，用 mask 把未来位置的注意力权重置为 $-\infty$
- **Cross-Attention**：Q 来自 Decoder 当前状态，K/V 来自 Encoder 所有时间步的隐状态——这就是 4.1 节讲的 Encoder-Decoder Attention，把源语言和目标语言的信息桥接起来

> **本章后续只用 Encoder**：情感分类、BERT 等理解类任务不需要逐词生成，只用 Encoder-only 架构就够了。Decoder-only（即去掉 Encoder、只保留 Masked Self-Attention 那一侧）是 GPT 系列的路线，用于文本生成，4.5 节会提到。

---

### Encoder 各部分说明

**1. Input Embedding**

词不能直接输入模型，先通过 Embedding 层把词 ID 转成浮点向量（形状：`seq_len × d_model`）。这张查找表是可学习的，训练中随模型一起更新。

---

**2. Positional Encoding（位置编码）**

Attention 本身是**置换不变**的——打乱词序，输出不变，因为它只计算词对之间的相似度，完全不感知"谁在谁前面"。为了注入位置信息，Transformer 对每个位置生成一个唯一的向量，**直接加到词向量上**：

$$PE_{(pos,\, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE_{(pos,\, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

用 sin/cos 函数的原因：不同位置的编码唯一，相邻位置的编码相似，而且这些编码是**固定的**（不需要训练），可以外推到训练时没见过的更长序列。

---

**3. Multi-Head Self-Attention（多头自注意力）**

单个 Attention 头只能学到一种"关注方式"。Multi-Head Attention 把 Q/K/V 分成 $h$ 组，每组独立做一次 Scaled Dot-Product Attention，再把结果拼接：

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \cdot W^O$$

这里是 **Self-Attention**——Q、K、V 全都来自同一个序列，每个词都和序列中所有其他词互相"查询"，从而捕捉句子内部任意距离的依赖关系。每个 head 可以学到不同维度的语义关系：有的关注句法依存，有的关注指代关系，有的关注位置相邻性。

每个子层后面都有**残差连接 + LayerNorm**：
- **残差连接**：让梯度直接流回输入，绕过子层，是训练深层网络的关键
- **LayerNorm**：对每个位置的特征维度做归一化，让分布稳定，加速收敛

---

**4. Feed-Forward Network（前馈网络，FFN）**

每个 Encoder Block 里，Attention 之后还有一个两层全连接网络：

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

中间维度通常是 $d_{model}$ 的 4 倍（如 $d_{model}=512$ 时，FFN 内部维度为 2048）。Attention 负责跨位置的信息聚合，FFN 负责对每个位置的特征做进一步的非线性变换，两者分工配合。

---

**5. 分类头**

Encoder 输出的是整个序列每个位置的向量（形状：`seq_len × d_model`）。对于分类任务，常见做法是对所有位置做 **Mean Pooling**（取均值），压缩成一个句子向量，再接一个线性层输出类别。BERT 则专门在句子开头加一个 `[CLS]` token，直接用它的输出做分类。

## 4.3 预训练模型与迁移学习

### 预训练的核心思路

自己从头训练一个 Transformer 的困境：IMDb 只有 25,000 条训练数据，模型的词义理解极其有限（"bank" 永远是同一个向量，无论是"银行"还是"河岸"）。

**预训练（Pre-training）** 的思路：先在海量无标注语料（Wikipedia、Common Crawl、书籍等，数十亿词）上做**自监督学习**，让模型学到丰富的语言知识；再用少量标注数据 **fine-tune** 到具体的下游任务。

类比：先让模型"读万卷书"，再"专项训练"。Fine-tune 通常只需要几百到几千条数据，因为通用语言知识已经内化在预训练权重中。

---

### BERT 架构

BERT（Bidirectional Encoder Representations from Transformers，Google 2018）是 NLP 预训练的里程碑。

**架构**：多层双向 Transformer Encoder（Base 版本：12 层，768 维，12 头，1.1 亿参数）

**为什么双向重要？**
- GPT 是从左到右的单向模型——预测当前词时只能看左边的上下文
- BERT 同时看左边和右边的上下文（双向），对于理解任务（分类、问答）效果更好
- 例如："I went to the **bank** to fish"——"bank" 是银行还是河岸，必须看右边的 "fish" 才能判断

**预训练任务1 — MLM（Masked Language Model）**：
随机遮住输入中 15% 的词（替换为 `[MASK]`），让模型预测被遮住的词。这迫使模型理解每个词与周围上下文的语义关系。

**预训练任务2 — NSP（Next Sentence Prediction）**：
给两个句子，判断它们在原文中是否连续。这让模型学到句间关系（用于问答、推理等任务）。

---

### 为什么 BERT 比从头训练好得多？

1. **数据规模**：BERT 见过 30 亿+ 词，而我们的 IMDb 训练集只有 500 万词
2. **双向上下文**：相同的词在不同上下文中得到不同的向量表示（动态词嵌入 vs 静态词嵌入）
3. **MLM 任务质量**：预测被遮住的词迫使模型建立深层语义理解，远比监督标签丰富

---

### HuggingFace Transformers

HuggingFace 提供了统一的 API，一行代码加载数百个预训练模型，是工业界标准工具：

```python
from transformers import pipeline
clf = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
clf('This movie is absolutely amazing!')
# [{'label': 'POSITIVE', 'score': 0.9998}]
```

**DistilBERT** 是 BERT 的知识蒸馏版本：参数量减少 40%（6层 vs 12层），推理速度提升 60%，而精度损失不到 3%。在资源受限的场景（移动端、实时推理）中广泛使用。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ══════════════════════════════════════════════════════════════════
# DistilBERT 情感分析演示
# ══════════════════════════════════════════════════════════════════
# 与 sharingsession_sentiment.ipynb 相同的测试句子
test_sentences = [
    'This movie is absolutely amazing, I loved every minute of it!',
    'This movie is not good at all, total waste of time.',
    'Not bad, I didn\'t expect to enjoy it but it was quite pleasant.',
    'I wanted to like this film but it was boring and disappointing.',
]

distilbert_acc_ref = 0.927   # DistilBERT fine-tuned on SST-2 的参考准确率（论文值）

try:
    from transformers import pipeline
    print('正在加载 DistilBERT（首次运行需下载约 250MB）...')
    clf = pipeline(
        'sentiment-analysis',
        model='distilbert-base-uncased-finetuned-sst-2-english',
        device=-1   # -1 = CPU
    )
    print('模型加载完成\n')

    results = clf(test_sentences)
    for sent, res in zip(test_sentences, results):
        label_cn = '正面 ✅' if res['label'] == 'POSITIVE' else '负面 ❌'
        print(f'  [{label_cn} {res["score"]:.3f}] "{sent[:55]}..."')

    # 用模型在4句测试样例上的平均置信度作为参考
    avg_conf = np.mean([r['score'] for r in results])
    print(f'\n4句测试样例平均置信度：{avg_conf:.3f}（仅供参考，非完整数据集准确率）')
    distilbert_demo = avg_conf

except Exception as e:
    print(f'[提示] 无法加载 DistilBERT（{type(e).__name__}）')
    print('  可能原因：transformers 未安装，或网络不通')
    print('  安装命令：pip install transformers')
    print()
    print('  伪代码示例（网络畅通时的运行结果）：')
    for sent in test_sentences:
        print(f'  [正面 ✅ 0.99]  "{sent[:55]}..."' if 'amazing' in sent or 'pleasant' in sent
              else f'  [负面 ❌ 0.02]  "{sent[:55]}..."')
    distilbert_demo = distilbert_acc_ref
    print(f'\n（使用参考值 {distilbert_acc_ref:.3f} 绘制对比图）')


# ══════════════════════════════════════════════════════════════════
# 四方法准确率对比 bar chart
# ══════════════════════════════════════════════════════════════════
cart_acc_ref  = 0.704
lstm_acc_ref  = 0.841
try:
    transformer_val = transformer_acc  # 来自上一 cell
except NameError:
    transformer_val = 0.83

methods = ['词袋 + CART', 'LSTM', 'Transformer', 'DistilBERT']
accs    = [cart_acc_ref, lstm_acc_ref, transformer_val, distilbert_demo]
colors  = ['#aec6cf', '#4a90d9', '#e07b54', '#6abf69']
labels  = ['传统方法', '第三章实战', '本章实战', '预训练模型']

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(methods, accs, color=colors, edgecolor='gray', width=0.5)
for bar, acc, lbl in zip(bars, accs, labels):
    ax.text(bar.get_x() + bar.get_width() / 2, acc + 0.01,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.text(bar.get_x() + bar.get_width() / 2, 0.02,
            lbl, ha='center', va='bottom', fontsize=8, color='gray')
ax.set_ylim(0, 1.05)
ax.set_ylabel('测试集准确率', fontsize=11)
ax.set_title('四种方法在 IMDb 情感分类上的准确率对比', fontsize=12)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='随机基准（0.5）')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print()
print('核心结论：')
print('  预训练模型（DistilBERT）在几乎零标注数据的情况下，')
print('  直接超越从头训练的 Transformer——这就是迁移学习的威力。')

## 4.4 生成模型概览

前几章的模型都是**判别模型**（Discriminative Model）：给定输入 $x$，预测标签 $y$，即建模 $P(y|x)$。

**生成模型**（Generative Model）则尝试学习数据本身的分布 $P(x)$，从而能够**生成新的、与训练数据相似的样本**。这是 AI 生成图像、视频、音乐、文本的基础。

---

### VAE（变分自编码器，2013）

**核心思想**：把输入 $x$ 映射到潜在空间的一个**分布**（而非单个点），从分布中采样 $z$，再解码重建 $x$。

- **Encoder**：$q_\phi(z|x)$，输出均值 $\mu$ 和标准差 $\sigma$
- **采样**：$z = \mu + \sigma \cdot \epsilon$，$\epsilon \sim \mathcal{N}(0, I)$（重参数化技巧，使梯度可以反向传播）
- **Decoder**：$p_	heta(x|z)$，从 $z$ 重建输入
- **损失**：重建误差（让输出接近输入）+ KL 散度（让潜在分布接近标准正态，保证空间连续）

**优势**：潜在空间是连续且有结构的，可以在两个样本的潜在向量之间插值，生成平滑的过渡样本。

---

### GAN（生成对抗网络，2014）

**核心思想**：两个网络对抗博弈。

- **Generator**（生成器）：接受随机噪声 $z$，生成假数据 $G(z)$，目标是让 Discriminator 无法分辨
- **Discriminator**（判别器）：给定输入，判断是真实数据还是生成数据，目标是准确区分
- **训练**：交替优化两者——D 努力识别假货，G 努力让假货以假乱真

理论上，博弈达到纳什均衡时，$G$ 生成的数据与真实数据分布完全一致。

**挑战**：
- **训练不稳定**：D 和 G 需要精心平衡，否则一方迅速碾压另一方
- **模式崩溃（Mode Collapse）**：G 发现只生成某几种样本就能骗过 D，从而丧失多样性

---

### Diffusion Models（扩散模型，2020 起）

**核心思想**：通过一个可逆的噪声过程学习数据分布。

- **前向过程**（固定）：逐步向真实数据 $x_0$ 加入高斯噪声，经过 $T$ 步后变成纯噪声 $x_T \sim \mathcal{N}(0, I)$
- **逆向过程**（学习）：训练一个神经网络 $\epsilon_	heta$，在每步预测并去除加入的噪声
- **生成**：从纯噪声 $x_T$ 出发，逐步去噪，最终得到真实样本

**优势**：训练稳定（没有对抗），生成质量高，支持条件引导（输入文本描述生成对应图像）。Stable Diffusion、DALL-E 2、Sora 的底层均基于此框架。

---

### 三种生成模型对比

| 维度 | VAE | GAN | Diffusion |
|------|-----|-----|-----------|
| **训练稳定性** | ✅ 稳定 | ⚠️ 易崩溃 | ✅ 稳定 |
| **生成质量** | ⚠️ 偏模糊 | ✅ 锐利 | ✅✅ 最高 |
| **生成速度** | ✅ 快（单步解码） | ✅ 快（单步推理） | ⚠️ 慢（多步去噪）|
| **可控生成** | ✅ 潜空间插值 | ⚠️ 较难 | ✅ 条件引导 |
| **模式多样性** | ✅ 好 | ⚠️ 模式崩溃风险 | ✅ 好 |
| **典型应用** | 表示学习、插值 | 图像生成（StyleGAN）| 图像/音频/视频生成 |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('三种生成模型的工作原理', fontsize=14, fontweight='bold', y=1.02)

# ── 通用工具 ───────────────────────────────────────────────────────────────────
def draw_box(ax, x, y, w, h, text, facecolor='#d0e8ff', edgecolor='#3a7fcc',
             fontsize=9.5, bold=False):
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                         boxstyle='round,pad=0.05',
                         facecolor=facecolor, edgecolor=edgecolor, linewidth=1.5)
    ax.add_patch(box)
    weight = 'bold' if bold else 'normal'
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight=weight, wrap=True)

def arrow(ax, x0, y0, x1, y1, color='#555555'):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))

# ══════════════════════════════════════════════════════════════════
# Panel 1 — VAE
# ══════════════════════════════════════════════════════════════════
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('VAE（变分自编码器）', fontsize=12, fontweight='bold', pad=10)

# 节点
draw_box(ax, 1.2, 5, 1.6, 1.0, 'Input\n$x$', '#ffe0b2', '#e08c00', bold=True)
draw_box(ax, 3.5, 5, 1.6, 1.0, 'Encoder\n$q_\\phi(z|x)$', '#d0e8ff', '#3a7fcc')
draw_box(ax, 5.8, 6.5, 1.4, 0.9, '$\\mu$', '#e8f5e9', '#388e3c')
draw_box(ax, 5.8, 3.5, 1.4, 0.9, '$\\sigma$', '#e8f5e9', '#388e3c')
draw_box(ax, 7.5, 5,   1.4, 1.0, '采样 $z$', '#f3e5f5', '#7b1fa2')
draw_box(ax, 9.0, 5,   1.4, 1.0, 'Decoder\n→ $\\hat{x}$', '#d0e8ff', '#3a7fcc')

# 箭头
arrow(ax, 2.0, 5, 2.7, 5)
arrow(ax, 4.3, 5.3, 5.1, 6.5)
arrow(ax, 4.3, 4.7, 5.1, 3.5)
arrow(ax, 6.5, 6.5, 6.8, 5.5)
arrow(ax, 6.5, 3.5, 6.8, 4.5)
arrow(ax, 8.2, 5, 8.3, 5)

# 注释
ax.text(5.0, 1.8, 'KL 散度约束\n让潜在分布 ~ N(0,I)', ha='center', fontsize=8.5,
        color='#7b1fa2', style='italic',
        bbox=dict(boxstyle='round', facecolor='#f3e5f5', edgecolor='#7b1fa2', alpha=0.7))
ax.text(5.0, 8.5, '重建损失\n让 $\\hat{x}$ ≈ $x$', ha='center', fontsize=8.5,
        color='#388e3c', style='italic',
        bbox=dict(boxstyle='round', facecolor='#e8f5e9', edgecolor='#388e3c', alpha=0.7))
ax.annotate('', xy=(1.2, 6.2), xytext=(9.0, 6.2),
            arrowprops=dict(arrowstyle='<->', color='#388e3c', lw=1.2, linestyle='--'))


# ══════════════════════════════════════════════════════════════════
# Panel 2 — GAN
# ══════════════════════════════════════════════════════════════════
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('GAN（生成对抗网络）', fontsize=12, fontweight='bold', pad=10)

draw_box(ax, 1.2, 7,   1.6, 1.0, '噪声 $z$\n$\\sim N(0,I)$', '#fff9c4', '#f9a825', bold=True)
draw_box(ax, 3.8, 7,   2.0, 1.0, 'Generator\n$G(z)$', '#d0e8ff', '#3a7fcc')
draw_box(ax, 6.5, 7,   1.6, 1.0, '假数据\n$G(z)$', '#ffccbc', '#e64a19')
draw_box(ax, 1.2, 3.5, 1.6, 1.0, '真实数据\n$x_{real}$', '#c8e6c9', '#388e3c', bold=True)
draw_box(ax, 6.5, 3.5, 1.6, 1.0, 'Discriminator\n$D(x)$', '#d0e8ff', '#3a7fcc')
draw_box(ax, 9.0, 5,   1.4, 1.0, '真 / 假\n输出', '#f3e5f5', '#7b1fa2')

arrow(ax, 2.0, 7,   2.8, 7)
arrow(ax, 4.8, 7,   5.7, 7)
arrow(ax, 7.3, 7,   7.3, 4.3)
arrow(ax, 2.0, 3.5, 5.7, 3.5)
arrow(ax, 7.3, 3.5, 8.3, 4.7)
ax.annotate('', xy=(3.8, 5.5), xytext=(7.3, 5.5),
            arrowprops=dict(arrowstyle='<-', color='#e64a19', lw=1.5, linestyle='dashed'))
ax.text(5.5, 5.0, 'G 反向更新\n（梯度来自 D）', ha='center', fontsize=8,
        color='#e64a19', style='italic')
ax.text(5.0, 1.5, '博弈均衡：$D(G(z)) = 0.5$', ha='center', fontsize=9,
        color='#7b1fa2', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='#f3e5f5', edgecolor='#7b1fa2', alpha=0.7))


# ══════════════════════════════════════════════════════════════════
# Panel 3 — Diffusion
# ══════════════════════════════════════════════════════════════════
ax = axes[2]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('Diffusion Model（扩散模型）', fontsize=12, fontweight='bold', pad=10)

steps_x = [1.0, 2.8, 4.6, 6.4, 8.2]
labels_d = ['$x_0$\n干净', '$x_1$', '$x_2$', '$x_{T-1}$', '$x_T$\n噪声']
alphas   = [1.0, 0.78, 0.56, 0.34, 0.12]
dot_colors = ['#1565c0', '#1976d2', '#42a5f5', '#90caf9', '#e3f2fd']

for i, (xpos, lbl, alph, col) in enumerate(zip(steps_x, labels_d, alphas, dot_colors)):
    circle = plt.Circle((xpos, 5.5), 0.7, color=col, ec='#1565c0', lw=1.5, alpha=max(alph, 0.2))
    ax.add_patch(circle)
    ax.text(xpos, 5.5, lbl, ha='center', va='center', fontsize=8, color='black' if alph > 0.4 else '#555')

# 省略号
ax.text(4.6, 5.5, '...', ha='center', va='center', fontsize=16, color='#888')

# 前向箭头（加噪）
for i in range(len(steps_x) - 1):
    if i != 2:
        arrow(ax, steps_x[i] + 0.75, 5.5, steps_x[i+1] - 0.75, 5.5, color='#1565c0')

# 注释：前向
ax.text(4.6, 7.8, '← 前向加噪（固定，逐步加入高斯噪声）', ha='center', fontsize=9,
        color='#1565c0', fontweight='bold')
ax.annotate('', xy=(1.0, 7.0), xytext=(8.2, 7.0),
            arrowprops=dict(arrowstyle='<-', color='#1565c0', lw=1.5))

# 逆向箭头（去噪）
ax.annotate('', xy=(8.2, 3.8), xytext=(1.0, 3.8),
            arrowprops=dict(arrowstyle='<-', color='#e64a19', lw=1.5))
ax.text(4.6, 2.8, '逆向去噪（学习，神经网络预测噪声） →', ha='center', fontsize=9,
        color='#e64a19', fontweight='bold')

# 底部说明
ax.text(4.6, 1.5, '代表应用：Stable Diffusion / DALL-E 2 / Sora', ha='center', fontsize=8.5,
        color='#555', style='italic',
        bbox=dict(boxstyle='round', facecolor='#fff9c4', edgecolor='#f9a825', alpha=0.7))

plt.tight_layout()
plt.show()

## 4.5 大语言模型简介

### GPT vs BERT

Transformer 催生了两条主流路线：

| | BERT | GPT 系列 |
|--|------|---------|
| **架构** | Encoder-only（双向） | Decoder-only（单向自回归） |
| **擅长** | 理解任务（分类、NER、问答） | 生成任务（续写、对话、代码） |
| **预训练目标** | MLM（填空，双向上下文） | 语言模型（预测下一个词，单向） |
| **推理方式** | 一次性编码整段输入 | 自回归逐词生成 |
| **代表模型** | BERT, RoBERTa, DeBERTa | GPT-2/3/4, LLaMA, Claude |

两者并非互斥：
- 需要**理解**（情感分析、信息抽取）→ BERT 类模型
- 需要**生成**（写作、对话、代码）→ GPT 类模型
- 现代大模型（如 GPT-4、Claude）通常在生成能力中也内置了强大的理解能力

---

### BERT 今天的商业应用

BERT 本身是基础模型，商业应用通常"藏在底层"，不像 GPT 那么被人直接感知到。

**搜索和排序（最大规模的商业应用）**：Google Search 从 2019 年起用 BERT 理解搜索词，Bing、百度等搜索引擎走类似路线。

**语义向量（今天最活跃的 Encoder 商业场景）**：OpenAI 的 `text-embedding-3-small/large`、Cohere Embed 等 Embedding API，本质上都是 Encoder 路线模型，把文本变成向量。这些向量被大量用于 **RAG（检索增强生成）**：先把文档库变成向量，用户提问时先检索相似文档，再喂给生成模型回答。

**后台分类服务**：垃圾邮件过滤、内容审核、情感分析 API，底层大多是 BERT 变体。

> 总结：BERT 路线模型今天主要以**向量嵌入**的形式存活，是 RAG 流水线的核心组件，而不是直接面向用户的对话界面。

---

### GPT 没有 Encoder，如何"理解"输入？

原始 Transformer（机器翻译那篇论文）需要单独的 Encoder，是因为要做**跨语言映射**——先把源语言句子编码成一套表示，再让 Decoder 通过 Cross-Attention 查询这套表示。

GPT 做的是单语言续写，不需要这种分离。它的 Decoder 同时承担了"理解输入"和"生成输出"两件事：

```
输入 prompt（所有用户消息）            生成 output（模型回复）
[token₁][token₂]...[tokenN]    →    [tokenN+1][tokenN+2]...
        ↑  Causal Self-Attention  ↑
        每个 token 可以看所有前面的 token
```

当模型处理到 prompt 最后一个 token 时，它已经通过多层 Self-Attention "看过"了整个 prompt。这最后一个 token 的隐藏状态，实际上就充当了 Encoder 输出"上下文向量"的角色——后续的生成从这里出发。

唯一的差异：GPT 的 Attention 是**单向**的（每个位置只看左边），而 BERT 是**双向**的（每个位置能看整句）。prompt 里靠前的 token 没法直接"看到"后面的词，但通过深度叠加的层，信息可以从前往后逐层传播，最终在最后一个位置汇聚。规模够大之后，这种单向 Attention 的局限基本可以被参数量和层数弥补。

所以 GPT 的"理解"不是一个独立的编码阶段，而是**和生成融合在一起的**——前几层处理 prompt，后几层开始输出，整个过程是一次前向传播完成的。

---

### Scaling Law（规模定律）

Kaplan 等（OpenAI，2020）发现，语言模型的性能（测试损失）与**参数量**、**数据量**、**计算量**之间存在幂律关系：

$$L(N) \propto N^{-0.076}, \quad L(D) \propto D^{-0.095}$$

这意味着：**更大的模型 + 更多数据 + 更多计算 = 更好的性能**，而且改进是可预测的。这个发现推动了从 BERT（1.1 亿参数）→ GPT-3（1750 亿参数）→ GPT-4（估计数万亿参数）的快速演进。

Chinchilla（DeepMind，2022）进一步指出：在固定计算预算下，参数量和训练数据量应**等比例增长**（早期 GPT-3 等模型数据不足），这是现代 LLM 训练的核心原则。

---

### 提示工程（Prompt Engineering）

大模型无需 fine-tune，通过设计输入提示就可以完成多种任务：

**Zero-shot**：直接给任务描述，无示例
```
任务：将以下句子翻译为英文。
输入：今天天气很好。
输出：
```

**Few-shot（In-Context Learning）**：给几个输入-输出示例，模型类比推理
```
今天天气很好。→ The weather is nice today.
我喜欢苹果。→ I like apples.
你好吗？→
```

**Chain-of-Thought（CoT，思维链）**：让模型"一步步思考"，显著提升复杂推理能力
```
Q: 小明有 3 个苹果，给了小红 1 个，又买了 5 个，现在有几个？
A: 让我一步步计算：
   初始：3 个
   给出：3 - 1 = 2 个
   购买：2 + 5 = 7 个
   答案：7 个
```

---

### RLHF（人类反馈强化学习）

ChatGPT 和 Claude 的核心对齐技术，分三步：

1. **监督微调（SFT）**：用人工标注的高质量对话 fine-tune 预训练语言模型
2. **奖励模型（Reward Model）**：人类对多个模型回复排序，训练一个能预测人类偏好的奖励函数
3. **PPO 强化学习**：用奖励模型的分数作为信号，用 PPO 算法进一步优化语言模型

目标：让模型的输出更**有帮助（Helpful）**、**无害（Harmless）**、**诚实（Honest）**——即 HHH 原则。

RLHF 使得同样参数量的模型，在遵循指令和避免有害输出方面大幅超越纯预训练模型。

In [ ]:
import torch

# ══════════════════════════════════════════════════════════════════
# GPT-2 文本续写演示
# ══════════════════════════════════════════════════════════════════
prompts = [
    'The future of artificial intelligence is',
    'Deep learning has transformed the field of',
    'In the next decade, machine learning will',
]

try:
    from transformers import pipeline, set_seed
    set_seed(42)
    print('正在加载 GPT-2（首次运行需下载约 500MB）...')
    generator = pipeline(
        'text-generation',
        model='gpt2',
        device=-1   # -1 = CPU
    )
    print('模型加载完成\n')

    print('=== GPT-2 文本续写演示 ===')
    for i, prompt in enumerate(prompts, 1):
        outputs = generator(
            prompt,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.8,
            pad_token_id=50256,  # GPT-2 的 EOS token，避免 padding 警告
            num_return_sequences=1
        )
        generated_text = outputs[0]['generated_text']
        print(f'--- Prompt {i} ---')
        print(f'  输入："{prompt}"')
        print(f'  续写：{generated_text[len(prompt):].strip()[:200]}')
        print()

except Exception as e:
    print(f'[提示] 无法加载 GPT-2（{type(e).__name__}）')
    print('  可能原因：transformers 未安装，或网络不通')
    print('  安装命令：pip install transformers')
    print()
    print('  伪代码示例（展示如何使用 GPT-2）：')
    print()
    print('  from transformers import pipeline, set_seed')
    print('  set_seed(42)')
    print('  generator = pipeline("text-generation", model="gpt2", device=-1)')
    print('  output = generator(')
    print('      "The future of artificial intelligence is",')
    print('      max_new_tokens=60,')
    print('      do_sample=True,')
    print('      temperature=0.8,')
    print('      pad_token_id=50256')
    print('  )')
    print('  # 输出示例：')
    print('  # "...increasingly intertwined with human decision-making,')
    print('  #   raising fundamental questions about autonomy and ethics."')


# ══════════════════════════════════════════════════════════════════
# Zero-shot vs Few-shot 对比示例
# ══════════════════════════════════════════════════════════════════
print()
print('=' * 55)
print('  Zero-shot vs Few-shot 提示工程对比')
print('=' * 55)
print()
print('=== Zero-shot ===')
print('Prompt:')
print('  "这部电影太棒了！请判断情感：正面 or 负面？"')
print()
print('  （期望输出：正面）')
print('  问题：模型可能不清楚输出格式，返回"这是一个正面评价"等冗长回答')
print()
print('=== Few-shot ===')
print('Prompt:')
print('  "糟糕透了"    → 负面')
print('  "非常精彩"    → 正面')
print('  "这部电影太棒了！" → ?')
print()
print('  （期望输出：正面，格式更稳定，准确率更高）')
print('  优势：示例明确了输出格式，模型通过类比直接给出答案')
print()
print('=== Chain-of-Thought ===')
print('Prompt:')
print('  "一家电影院有 5 个放映厅，每厅 120 座，今天卖出 80% 的票，')
print('   但有 30 人退票，请问今天实际观影人数是？')
print('   请一步步计算。"')
print()
print('  模型输出：')
print('  "总座位数：5 × 120 = 600')
print('   卖出：600 × 80% = 480')
print('   退票后：480 - 30 = 450 人"')
print()
print('  → 加入"请一步步计算"后，复杂数学推理准确率大幅提升')
print()
print('=' * 55)
print()
print('【生产建议】')
print('  - GPT-2（117M 参数，2019 年）在现代标准下属于小模型，')
print('    文本连贯性和指令遵循能力有限，主要用于教学演示。')
print()
print('  - 实际生产环境推荐使用现代大模型 API：')
print('    • Anthropic Claude API：claude-opus-4 / claude-sonnet-4')
print('    • OpenAI API：gpt-4o / gpt-4o-mini')
print()
print('  - 这些现代模型在指令遵循、推理、安全性上')
print('    比 GPT-2 有数量级的提升，且通过 API 调用无需本地 GPU。')